# Analyze a retrieval run: recall@fetch, per-question

The repeatable 4-step method for comparing indexes on a `run_retrieval.py` run
(e.g. Markdown vs PDF-outline). See `reports/runbook-representation-study.md`.

1. **Summary table** — recall + content-tokens + unmappable, per index.
2. **Per-question pivot** — find *where* the indexes diverge (the gap is the story).
3. **Drill-down** — read what a divergent question actually fetched + answered.
4. **Interpretation checklist** — real miss vs metric artifact.

**Kernel note:** pure stdlib — runs on any Python 3 kernel, no pandas needed. It
shells out to `scripts/score_recall.py` via the repo's `.venv` so the scorer stays
the single source of truth.

## Config

Set `RUN` to a run timestamp, or leave it `None` to auto-pick the newest run.
Everything below keys off this cell.

In [ ]:
from pathlib import Path

def find_repo(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "scripts" / "score_recall.py").exists():
            return p
    raise RuntimeError("run this notebook from inside the repo")

REPO = find_repo()

RUN = None   # e.g. "20260715T174748Z"; None = newest run with a run.json
if RUN is None:
    dirs = [p for p in (REPO / "runs").iterdir() if p.is_dir() and (p / "run.json").exists()]
    RUN = sorted(p.name for p in dirs)[-1]

RUN_DIR = REPO / "runs" / RUN
QUESTIONS = REPO / "evaluations" / "questions-rfc9110.csv"   # gold set for this corpus
VENV_PY = REPO / ".venv" / "bin" / "python"
print("repo :", REPO)
print("run  :", RUN_DIR)
print("gold :", QUESTIONS.name)

## Step 1 — Summary table

Run the scorer (free; no LLM). It reprints the aggregate table **and** regenerates
`runs/<RUN>/recall.csv`, which Steps 2–3 read.

Columns to read:
- **`recall`** — fraction of gold sections the retriever fetched. The metric.
- **`content_tok`** — how much text it pulled to get there. The *cost* of the recall.
- **`unmap_gold`** — **check this first.** Nonzero = that index's tree lost some gold
  section labels, so the comparison is unfair (e.g. an LLM-inferred tree). Zero = fair.

In [ ]:
import subprocess

res = subprocess.run(
    [str(VENV_PY), str(REPO / "scripts" / "score_recall.py"),
     "--run", str(RUN_DIR), "--questions", str(QUESTIONS)],
    capture_output=True, text=True)
print(res.stdout or res.stderr)

## Step 2 — Per-question pivot (find *where* they differ)

The aggregate hides the story. A 0.09 mean gap that's really 4 questions is a totally
different finding than 0.09 spread evenly. This pivots recall by question across every
index in the run and sorts by the largest gap, so the divergences float to the top.

Works for any 2+ indexes — index names are read from the run, not hardcoded.

In [ ]:
import csv

rows = [r for r in csv.DictReader((RUN_DIR / "recall.csv").open()) if r["absence"] == "False"]
indexes = sorted({r["index"] for r in rows})

byq = {}
for r in rows:
    if r["recall"] == "":
        continue
    byq.setdefault((r["qid"], r["category"]), {})[r["index"]] = float(r["recall"])

table = []
for (qid, cat), rec in byq.items():
    vals = [rec.get(ix) for ix in indexes]
    if any(v is None for v in vals):
        continue
    table.append((max(vals) - min(vals), qid, cat, vals))
table.sort(reverse=True)   # biggest gap first

def short(ix):
    return ix.replace("IDX-", "").replace("-rfc9110", "")

hdr = f"{'qid':6}{'category':<28}" + "".join(f"{short(ix):>16}" for ix in indexes) + f"{'gap':>7}"
print(hdr); print("-" * len(hdr))
for gap, qid, cat, vals in table:
    flag = "  <-- differ" if gap > 0.01 else ""
    print(f"{qid:6}{cat:<28}" + "".join(f"{v:>16.2f}" for v in vals) + f"{gap:>7.2f}{flag}")

gap_qids = [qid for gap, qid, *_ in table if gap > 0.01]
print(f"\n{len(gap_qids)} question(s) differ: {gap_qids}")

## Step 3 — Drill-down: what did it actually fetch + answer?

The habit that separates a real analysis from a number. For each divergent question,
read **what it fetched** (wide line-ranges vs discrete node-ids → the mechanism) and
**whether the answer is actually right** (catches metric artifacts — a page-addressed
arm can answer correctly while scoring low, because a coarse node bundles neighbor text).

In [ ]:
import json

run = json.load((RUN_DIR / "run.json").open())

def inspect(qid, chars=400):
    q = next((r for r in run["results"] if r["qid"] == qid), None)
    print(f"===== {qid} =====")
    for r in run["results"]:
        if r["qid"] != qid:
            continue
        pulls = [c["args"].get("pages") for c in r["tool_calls"] if c["tool"] == "get_page_content"]
        print(f"\n[{r['index_id']}]  fetched = {pulls}")
        print("  " + " ".join((r.get("answer") or "").split())[:chars])

# auto-inspect the biggest-gap question; call inspect("RB3") etc. for any other
if gap_qids:
    inspect(gap_qids[0])
else:
    print("no divergent questions in this run")

## Step 4 — Interpretation checklist

Run through this for each divergence before writing it up:

1. **`unmap_gold` zero on both?** (Step 1) → fair comparison. If not, stop — fix the index.
2. **Broad or a few questions?** (Step 2) → a small mean gap that's really N questions is a
   *specific failure mode*, not a general verdict.
3. **Real miss or metric artifact?** (Step 3) → read the *answer*, not just the recall.
   `recall@fetch` counts whether the gold *node* was fetched, so it **under-credits**
   coarse/page-addressed arms that pack more per fetch.
4. **Pair recall with `content_tok`.** High recall bought with huge fetches is not the same
   win as high recall on few tokens — that trade is often the real headline.

Write the conclusion as *"index A costs X points of recall vs B, concentrated in
\<categories\>, while using Y% \<more/fewer\> content tokens"* — not "A is better than B."